# 📘 Capítulo 3 — Categorical Variables (Variáveis Categóricas)
- Como transformar texto em informação útil para modelos de Machine Learning
- Hybrid Learning Notebook — Study + Reference + Hands-on

## 📚 Curso Intermediate Machine Learning

- Variáveis categóricas aparecem em praticamente todos os datasets reais:
    - tipos de imóveis
    - materiais de construção
    - bairros
    - métodos de venda
    - cores
    - categorias de produto
    - etc.

<br>

> Modelos de Machine Learning **não conseguem trabalhar diretamente com texto**. <br>
*Por isso, precisamos transformar essas categorias em números — e existem várias formas de fazer isso.*

<br>

> #### 🎯 Objetivos
- Neste capítulo, você aprenderá:
    - o que é uma variável categórica  
    - diferença entre variáveis **nominais** e **ordinais**  
    - três abordagens para tratá-las  
    - quando usar cada abordagem  
    - como comparar desempenho entre elas  
    - como evitar erros comuns  

    Este notebook segue **exatamente o conteúdo do curso Kaggle**, mas com explicações mais ricas e estruturadas.

---

### 🧩 Estrutura do Projeto

- Este notebook faz parte do ecossistema:
```text
    ML/
        kaggle/
            intermediate-ml/
                data/raw/        → datasets originais (train.csv, test.csv)
                data/processed/  → datasets limpos
                notebooks/       → notebooks de estudo
                models/          → modelos treinados
                outputs/         → arquivos de saída (submission.csv, gráficos)
                docs/            → documentação estilo livro
```

---

### 🟦 Glossário Técnico

- Consulta Rápida

    - **Variável categórica** — variável cujo valor pertence a um conjunto limitado de categorias (ex.: "Casa", "Apartamento").  
    - **Variável nominal** — categorias sem ordem natural (ex.: "Vermelho", "Azul").  
    - **Variável ordinal** — categorias com ordem natural (ex.: "Baixo", "Médio", "Alto").  
    - **Cardinalidade** — número de categorias distintas em uma variável.  
    - **Encoding** — processo de transformar categorias em números.  
    - **Ordinal Encoding** — substitui cada categoria por um número inteiro.  
    - **One-Hot Encoding** — cria uma coluna para cada categoria, com valores 0/1.  
    - **Sparse Matrix** — matriz com muitos zeros (comum em one-hot encoding).  
    - **Leakage** — quando informações do futuro “vazam” para o treino.  


---


### 🟩 Mini‑Referência (API Style)

- Comandos Essenciais

    - Identificar colunas categóricas
        - df.select_dtypes(include=['object', 'string'])

    - Ordinal Encoding
        - from sklearn.preprocessing import OrdinalEncoder
        - encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

    - One-Hot Encoding
        - from sklearn.preprocessing import OneHotEncoder
        - OH = OneHotEncoder(handle_unknown='ignore', sparse=False)

    - Concatenar DataFrames
        - pd.concat([df1, df2], axis=1)

    - Remover colunas
        - df.drop(cols, axis=1)

---


### 🟧 Explicação Conceitual (Book Style)

> Por que variáveis categóricas são um desafio?

- Modelos de Machine Learning trabalham com **números**, não com texto.

    - Se você tentar treinar um modelo com colunas contendo strings, receberá um erro.  
    - Por isso, precisamos converter texto em números — mas **a forma como fazemos isso importa**.

<br>

Existem três abordagens principais:

---

#### 1) Dropar variáveis categóricas  
- Simples, rápido, mas perde informação.  
- Funciona apenas quando as categorias não são relevantes.

---

#### 2) Ordinal Encoding  
- Cada categoria vira um número inteiro.

    Exemplo:
    - "Baixo" → 0  
    - "Médio" → 1  
    - "Alto" → 2  

- Funciona bem quando existe **ordem natural**.  
- Funciona bem com **modelos baseados em árvores** (Random Forest, XGBoost).

---

#### 3) One-Hot Encoding
- Cria uma coluna para cada categoria.

    Exemplo:
    - Cor = ["Vermelho", "Azul", "Verde"]

- Vira:

```text
    | Vermelho | Azul     | Verde    |
    |----------|----------|----------|
    | 1        | 0        | 0        |
    | 0        | 1        | 0        |
    | 0        | 0        | 1        | 
```

- Não assume ordem.  
- Funciona bem para variáveis **nominais**.  
- Pode gerar muitas colunas se houver alta cardinalidade.

<br>

---

#### Quando usar cada abordagem?

```text
    | Situação                  | Melhor abordagem       |
    |---------------------------|------------------------|
    | Variável ordinal          | Ordinal Encoding       |
    | Variável nominal          | One-Hot Encoding       |
    | Muitas categorias (> 15)  | Evitar One-Hot         |
    | Modelo baseado em árvores | Ordinal ou One-Hot     |
    | Modelo linear             | One-Hot (quase sempre) |
```

<br>

---


### 🟨 Importar Bibliotecas + Carregar Dataset Ames Housing

In [2]:
import sys
from pathlib import Path

# Caminho absoluto do notebook
notebook_path = Path().resolve()

# Sobe diretórios até encontrar config.py
for parent in notebook_path.parents:
    if (parent / "config.py").exists():
        sys.path.append(str(parent))
        break

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from config import get_data_path, get_output_path

DATA_PATH = get_data_path()
OUTPUT_PATH = get_output_path()

### 🟪 Carregar Dados + Validação Automática

> Dataset Ames Housing <br>
*Este é o dataset oficial do curso Kaggle Intermediate ML.*

- Ele contém 79 variáveis explicativas sobre casas, incluindo:
    - materiais  
    - condições  
    - tipos de cômodos  
    - vizinhança  
    - qualidade  
    - ano de construção  
    - e muito mais  

> É um dataset rico e limpo, ideal para aprendizado.

In [4]:
# Carregar dataset
X_full = pd.read_csv(DATA_PATH + "train.csv", index_col="Id")

# Validar presença da coluna SalePrice
assert "SalePrice" in X_full.columns, "Dataset incorreto: SalePrice não encontrada."

# Remover linhas com SalePrice ausente
X_full.dropna(axis=0, subset=["SalePrice"], inplace=True)

# Separar target
y = X_full.SalePrice

# Remover target do dataset
X = X_full.drop(["SalePrice"], axis=1)

# Split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size=0.8, random_state=0
)

# Converter colunas string → object (compatibilidade com Kaggle)
for col in X_train.columns:
    if X_train[col].dtype == "string" or X_train[col].dtype == "str":
        X_train[col] = X_train[col].astype("object")
        X_valid[col] = X_valid[col].astype("object")

X_train.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,
619,20,RL,90.0,11694,Pave,NaN,Reg,Lvl,AllPub,Inside,...,260,0,NaN,NaN,NaN,0,7,2007,New,Partial
871,20,RL,60.0,6600,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,8,2009,WD,Normal
93,30,RL,80.0,13360,Pave,Grvl,IR1,HLS,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,8,2009,WD,Normal
818,20,RL,NaN,13265,Pave,NaN,IR1,Lvl,AllPub,CulDSac,...,0,0,NaN,NaN,NaN,0,7,2008,WD,Normal
303,20,RL,118.0,13704,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,1,2006,WD,Normal


### 🟫 Identificar Variáveis Categóricas

>  Identificando colunas categóricas <br>
Variáveis categóricas geralmente têm dtype "object".

In [5]:
object_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Variáveis categóricas detectadas:")
object_cols

Variáveis categóricas detectadas:


['MSZoning',
 'Street',
 'Alley',
 'LotShape',
 'LandContour',
 'Utilities',
 'LotConfig',
 'LandSlope',
 'Neighborhood',
 'Condition1',
 'Condition2',
 'BldgType',
 'HouseStyle',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Exterior2nd',
 'MasVnrType',
 'ExterQual',
 'ExterCond',
 'Foundation',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Heating',
 'HeatingQC',
 'CentralAir',
 'Electrical',
 'KitchenQual',
 'Functional',
 'FireplaceQu',
 'GarageType',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PavedDrive',
 'PoolQC',
 'Fence',
 'MiscFeature',
 'SaleType',
 'SaleCondition']

### ⭐ Função de Avaliação (MAE)

In [5]:
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

### ⭐ Abordagem 1 — Dropar Variáveis Categóricas

> Simples, rápido, mas perde informação valiosa.


In [6]:
drop_X_train = X_train.drop(object_cols, axis=1)
drop_X_valid = X_valid.drop(object_cols, axis=1)

print("MAE (Drop):")
score_dataset(drop_X_train, drop_X_valid, y_train, y_valid)

MAE (Drop):


18315.570707762556

### ⭐ Abordagem 2 — Ordinal Encoding

- Transforma cada categoria em um número inteiro.
- Funciona bem com modelos baseados em árvores.

In [7]:
label_X_train = X_train.copy()
label_X_valid = X_valid.copy()

ordinal_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

label_X_train[object_cols] = ordinal_encoder.fit_transform(X_train[object_cols])
label_X_valid[object_cols] = ordinal_encoder.transform(X_valid[object_cols])

print("MAE (Ordinal Encoding):")
score_dataset(label_X_train, label_X_valid, y_train, y_valid)

MAE (Ordinal Encoding):


17299.950410958903

### ⭐ Abordagem 3 — One-Hot Encoding

- Cria uma coluna para cada categoria.
- Não assume ordem.
- Funciona bem para variáveis nominais.

In [8]:
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[object_cols]))

OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

print("MAE (One-Hot Encoding):")
score_dataset(OH_X_train, OH_X_valid, y_train, y_valid)


MAE (One-Hot Encoding):


17430.58839041096

### 📊 10. Comparação Final

### 🧠 Conclusão

> Conclusão

- Dropar variáveis categóricas geralmente é a pior opção.  
- Ordinal Encoding funciona bem para variáveis ordinais e modelos baseados em árvores.  
- One-Hot Encoding é geralmente a melhor opção para variáveis nominais.  
- A escolha depende da natureza da variável e do modelo utilizado.

Você agora domina as três técnicas fundamentais para tratar variáveis categóricas.